# 01 — Prepare Data (Colab → Google Drive)

Chạy tiền xử lý ChatML cho Task 1/2/3 trên Colab và **ghi toàn bộ output lên Google Drive**.

Drive folder đích: [`data`](https://drive.google.com/drive/u/2/folders/1P31-GtTh5GHcUxO7TJhftSONuBTvvKwI)  
Folder ID: `1P31-GtTh5GHcUxO7TJhftSONuBTvvKwI`

**Cấu trúc sẽ tạo trên Drive:**
```text
data/                          ← folder Drive ở trên
├── raw/                       ← đặt sample_dataset hoặc full dataset vào đây
│   ├── exams.json
│   ├── task1_grading.json
│   ├── task2_error_taxonomy.json
│   ├── task3_feedback.json
│   ├── label_space.json
│   └── submissions/
├── processed/                 ← train_unified.jsonl, task*_*.jsonl, ...
├── splits/                    ← train_ids.json, val_ids.json
└── outputs/                   ← eda_summary.json
```

## 0. Cấu hình đường dẫn

In [9]:
# === CHỈNH Ở ĐÂY NẾU CẦN ===
# Path tới folder Drive đích (sau khi Add shortcut to My Drive nếu là shared folder).
# Link: https://drive.google.com/drive/folders/1P31-GtTh5GHcUxO7TJhftSONuBTvvKwI
DRIVE_DATA_DIR = "/content/drive/MyDrive/LLM/Final Project/data"

# Repo public — Colab sẽ git clone vào /content/llm-final-project
REPO_URL = "https://github.com/tiendinhquang2104/llm-final-project.git"
REPO_DIR_ON_DRIVE = ""  # hoặc path code đã upload lên Drive

SEED = 42
VAL_RATIO = 0.2


## 1. Mount Google Drive & resolve folder

In [10]:
from pathlib import Path
import os, json, subprocess, sys

from google.colab import drive
drive.mount("/content/drive", force_remount=False)


def resolve_drive_data(preferred: str) -> Path:
    """Chỉ dùng path đã mount — KHÔNG gọi auth.authenticate_user() (tránh lỗi Credentials propagation)."""
    p = Path(preferred)
    if p.exists():
        return p.resolve()

    found = []
    roots = [Path("/content/drive/MyDrive"), Path("/content/drive/Shareddrives")]
    for root in roots:
        if not root.exists():
            continue
        for d in root.rglob("data"):
            if not d.is_dir():
                continue
            try:
                rel = d.relative_to(root)
            except ValueError:
                continue
            if len(rel.parts) > 3:
                continue
            found.append(d)
            if len(found) >= 20:
                break

    for d in found:
        if (d / "raw").exists() or (d / "exams.json").exists() or (d / "processed").exists():
            return d.resolve()
    if found:
        print("Các ứng viên 'data' trên Drive:")
        for d in found:
            print(" -", d)
        return found[0].resolve()

    p.mkdir(parents=True, exist_ok=True)
    print(f"Đã tạo folder mới: {p}")
    print("Nếu đây không đúng folder shared, hãy:")
    print("  1) Mở https://drive.google.com/drive/folders/1P31-GtTh5GHcUxO7TJhftSONuBTvvKwI")
    print("  2) Add shortcut to Drive → My Drive")
    print("  3) Sửa DRIVE_DATA_DIR cho khớp path thật, rồi chạy lại cell này")
    return p.resolve()


DRIVE_DATA = resolve_drive_data(DRIVE_DATA_DIR)
RAW_DIR = DRIVE_DATA / "raw"
PROCESSED_DIR = DRIVE_DATA / "processed"
SPLITS_DIR = DRIVE_DATA / "splits"
OUTPUTS_DIR = DRIVE_DATA / "outputs"
DRIVE_FOLDER_ID = "1P31-GtTh5GHcUxO7TJhftSONuBTvvKwI"

for d in (RAW_DIR, PROCESSED_DIR, SPLITS_DIR, OUTPUTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("DRIVE_DATA   =", DRIVE_DATA)
print("RAW_DIR      =", RAW_DIR)
print("PROCESSED_DIR=", PROCESSED_DIR)
print("exists?      =", DRIVE_DATA.exists())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DRIVE_DATA   = /content/drive/MyDrive/LLM/Final Project/data
RAW_DIR      = /content/drive/MyDrive/LLM/Final Project/data/raw
PROCESSED_DIR= /content/drive/MyDrive/LLM/Final Project/data/processed
exists?      = True


## 2. Lấy mã nguồn repo lên Colab

Ưu tiên: clone `REPO_URL` → nếu trống thì dùng `REPO_DIR_ON_DRIVE` → nếu vẫn trống thì upload thủ công vào `/content/llm-final-project`.

In [11]:
LOCAL_REPO = Path("/content/llm-final-project")

if REPO_URL:
    if LOCAL_REPO.exists():
        print("Repo đã có:", LOCAL_REPO)
    else:
        subprocess.check_call(["git", "clone", REPO_URL, str(LOCAL_REPO)])
elif REPO_DIR_ON_DRIVE:
    src = Path(REPO_DIR_ON_DRIVE)
    assert src.exists(), f"Không thấy code trên Drive: {src}"
    if LOCAL_REPO.exists():
        print("Repo local đã có:", LOCAL_REPO)
    else:
        subprocess.check_call(["cp", "-a", str(src), str(LOCAL_REPO)])
        print("Copied from Drive →", LOCAL_REPO)
else:
    # Thử tìm notebook đang chạy từ bản copy trong Drive / upload sẵn
    candidates = [
        LOCAL_REPO,
        Path("/content/drive/MyDrive/llm-final-project"),
        DRIVE_DATA.parent / "llm-final-project",
    ]
    found = next((p for p in candidates if (p / "scripts" / "prepare_data.py").exists()), None)
    if found is None:
        raise FileNotFoundError(
            "Chưa có source code. Điền REPO_URL hoặc REPO_DIR_ON_DRIVE ở cell cấu hình, "
            "hoặc upload repo vào /content/llm-final-project."
        )
    if found != LOCAL_REPO:
        if not LOCAL_REPO.exists():
            subprocess.check_call(["cp", "-a", str(found), str(LOCAL_REPO)])
    print("Using repo:", LOCAL_REPO)

os.chdir(LOCAL_REPO)
sys.path.insert(0, str(LOCAL_REPO))
print("cwd:", Path.cwd())
assert (LOCAL_REPO / "scripts" / "prepare_data.py").exists()

cwd: /content/llm-final-project


## 3. Đặt raw dataset vào Drive `data/raw/`

Cần có `exams.json` (và các file task + `submissions/`) **trực tiếp** trong `data/raw/`  
hoặc trong `data/raw/sample_dataset/`.

Cell dưới chỉ kiểm tra; nếu thiếu thì upload bộ sample/full lên Drive (kéo thả vào folder `raw`).

In [12]:
from src.data.loader import resolve_dataset_root

try:
    DATA_DIR = resolve_dataset_root(RAW_DIR)
    print("✓ Raw dataset OK:", DATA_DIR)
    print("  files:", sorted(p.name for p in DATA_DIR.iterdir() if p.is_file())[:10])
except FileNotFoundError as e:
    print("✗", e)
    print()
    print("Hãy upload dataset vào:", RAW_DIR)
    print("Link folder:", f"https://drive.google.com/drive/folders/{DRIVE_FOLDER_ID}")
    print("Cần: exams.json, task1_grading.json, task2_error_taxonomy.json, task3_feedback.json, submissions/")
    raise

✗ Cannot find exams.json under /content/drive/MyDrive/LLM/Final Project/data/raw. Point --data_dir at the folder that contains exams.json, task1_grading.json, and submissions/.

Hãy upload dataset vào: /content/drive/MyDrive/LLM/Final Project/data/raw
Link folder: https://drive.google.com/drive/folders/1P31-GtTh5GHcUxO7TJhftSONuBTvvKwI
Cần: exams.json, task1_grading.json, task2_error_taxonomy.json, task3_feedback.json, submissions/


FileNotFoundError: Cannot find exams.json under /content/drive/MyDrive/LLM/Final Project/data/raw. Point --data_dir at the folder that contains exams.json, task1_grading.json, and submissions/.

## 4. Chạy preprocess → ghi lên Drive

In [ ]:
from src.data.preprocess import build_unified_jsonl

# Ghi processed vào Drive; splits cũng sẽ vào Drive data/splits vì output_dir tên 'processed' và parent là 'data'
paths = build_unified_jsonl(
    data_dir=DATA_DIR,
    output_dir=PROCESSED_DIR,
    splits_dir=SPLITS_DIR,
    val_ratio=VAL_RATIO,
    seed=SEED,
    check_leakage=True,
)

print("=== Written ===")
for k, p in paths.items():
    size = p.stat().st_size if p.exists() else 0
    print(f"{k:18s} {p}  ({size:,} bytes)")

manifest = json.loads(paths["manifest"].read_text(encoding="utf-8"))
print("\nmanifest:", json.dumps(manifest, ensure_ascii=False, indent=2))

## 5. EDA summary → Drive `data/outputs/`

In [ ]:
import subprocess

eda_out = OUTPUTS_DIR / "eda_summary.json"
subprocess.check_call([
    sys.executable, "scripts/eda_report.py",
    "--data_dir", str(DATA_DIR),
    "--out", str(eda_out),
])
print(eda_out.read_text(encoding="utf-8")[:1500])

## 6. Smoke-check + đường dẫn dùng cho training sau này

In [ ]:
train = PROCESSED_DIR / "train_unified.jsonl"
val = PROCESSED_DIR / "val_unified.jsonl"
assert train.exists() and val.exists()

n_train = sum(1 for _ in train.open(encoding="utf-8"))
n_val = sum(1 for _ in val.open(encoding="utf-8"))
row = json.loads(train.open(encoding="utf-8").readline())
assert "<|im_start|>" in row["text"]
assert row["task"] in {"task1", "task2", "task3"}

print(f"OK train={n_train} val={n_val}")
print()
print("Dùng các path này ở notebook train:")
print("  TRAIN_JSONL =", train)
print("  VAL_JSONL   =", val)
print("  SPLITS_DIR  =", SPLITS_DIR)
print("  DRIVE_DATA  =", DRIVE_DATA)
print()
print("Mở Drive:", f"https://drive.google.com/drive/folders/{DRIVE_FOLDER_ID}")